In [2]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [6]:
def step_1(state: CrashState)-> CrashState:
    print("step 1 executed")
    return {"step1": "done", "input": state['input']}
def step_2(state: CrashState)-> CrashState:
    print("step 2 hanging")
    time.sleep(30)
    return {"step2":"done"}
def step_3(state: CrashState)->CrashState:
    print("step3 done")
    return {"step3":"done"}

In [7]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [9]:

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
step 1 executed
step 2 hanging
step3 done
